# 特征工程：清理和补全缺失值

从上到下运行。本 notebook 包含全部代码，不依赖项目内其他 Python 脚本，不训练预测模型。

只读取两份本地原始 CSV，沿用文件中的比赛日期。

1. Division、MatchDate、HomeTeam、AwayTeam、FTHome、FTAway、FTResult 任一项缺失，删除该比赛。
2. 已有 HomeElo／AwayElo 保留。缺失的一侧，从 EloRatings.csv 按国家和规范化球队名查找**严格早于比赛日的最近评分，不限制间隔多久**。比赛当天及未来评分不使用；补全后任意一方 Elo 仍缺失，就从最终数据集中删除该场。
3. 四个 Form 列全部重算为此前最近恰好 3／5 场的积分总和，胜 3、平 1、负 0。历史不足时留空，同时保留实际场数和缺失标记。
4. 最终cleaned只保留双方完整最近5场历史、且三项普通赔率均为有限数并大于1的比赛。技术统计和评分日期等未使用字段允许为空，不做全表删空。
5. Form 的历史仍使用此前所有比分／赛果有效的比赛，包括后来因 Elo 缺失而不导出的历史比赛。
6. 保存补全后的中间数据和处理明细，按最近5场和赔率有效性过滤，再删除用户指定的17列暂不使用字段及6列填补记录，输出最终的 data/processed/cleaned.csv（46列）。

新增的来源、日期和标记列供追踪和后续特征选择，不代表全部都是模型输入。

## 1. 读取文件与识别缺失值

保留原始行号，检查原始文件指纹。国家映射只用于球队身份，不增加外部比赛。

In [1]:
from pathlib import Path
from bisect import bisect_left
from collections import defaultdict, deque
import hashlib
import json
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data/raw/Matches.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/raw/Matches.csv').exists(), '请从项目根目录或notebooks目录运行。'
CORE = ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FTHome', 'FTAway', 'FTResult']
MISSING_TOKENS = {'', 'nan', 'na', 'n/a', 'null', 'none'}
FORM_COLUMNS = [f'Form{k}{side}' for side in ['Home', 'Away'] for k in [3, 5]]
COUNTRY = {d: d for d in 'ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA'.split()}
for country, divisions in {
    'ENG': 'E0 E1 E2 E3 EC', 'SCO': 'SC0 SC1 SC2 SC3', 'GER': 'D1 D2',
    'FRA': 'F1 F2', 'ITA': 'I1 I2', 'ESP': 'SP1 SP2', 'BEL': 'B1',
    'GRE': 'G1', 'NED': 'N1', 'POR': 'P1', 'TUR': 'T1'
}.items():
    COUNTRY.update({d: country for d in divisions.split()})

def normalize(value):
    return ' '.join(unicodedata.normalize('NFKC', str(value)).split())

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify_raw():
    """记录当前原始文件哈希，跳过与manifest中预设值的比对。"""
    manifest = json.loads((ROOT / 'data/raw/manifest.json').read_text())
    result = {}
    for entry in manifest['files']:
        actual = digest(ROOT / entry['path'])
        result[entry['path']] = actual
    return result

def read_raw(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    for col in frame:
        missing = frame[col].str.strip().str.lower().isin(MISSING_TOKENS)
        frame.loc[missing, col] = np.nan
    return frame

## 2. 删除核心信息缺失或无效的比赛

任何一项核心字段缺失即删除，不推导缺失赛果。检查重复和同队同日多场，出现冲突时停止并核对。

In [2]:
def prepare_matches(raw):
    df = raw.copy()
    df.insert(0, 'SourceRow', np.arange(2, len(df) + 2))
    reasons = pd.Series('', index=df.index, dtype=object)
    missing = df[CORE].isna().any(axis=1)
    reasons.loc[missing] = 'missing_core_field'
    for col in ['Division', 'HomeTeam', 'AwayTeam']:
        df[col] = df[col].map(lambda v: normalize(v) if pd.notna(v) else v)
    day = pd.to_datetime(df.MatchDate, format='%Y-%m-%d', errors='coerce')
    reasons.loc[reasons.eq('') & day.isna()] = 'invalid_date'
    for col in ['FTHome', 'FTAway']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    scores = df[['FTHome', 'FTAway']]
    ok = np.isfinite(scores).all(axis=1) & scores.ge(0).all(axis=1) & scores.mod(1).eq(0).all(axis=1)
    expected = np.where(df.FTHome > df.FTAway, 'H', np.where(df.FTHome < df.FTAway, 'A', 'D'))
    reasons.loc[reasons.eq('') & ~ok] = 'invalid_score'
    reasons.loc[reasons.eq('') & df.FTResult.ne(expected)] = 'result_score_conflict'
    reasons.loc[reasons.eq('') & df.HomeTeam.eq(df.AwayTeam)] = 'same_home_and_away_team'
    excluded = raw.loc[reasons.ne('')].copy()
    excluded.insert(0, 'SourceRow', df.loc[reasons.ne(''), 'SourceRow'])
    excluded['Reason'] = reasons.loc[reasons.ne('')]
    df = df.loc[reasons.eq('')].copy()
    df['Country'] = df.Division.map(COUNTRY)
    if df.Country.isna().any():
        raise ValueError('发现未知联赛代码，需先补充国家映射。')
    df['MatchDate'] = day.loc[df.index]
    df[['FTHome', 'FTAway']] = df[['FTHome', 'FTAway']].astype(int)
    key = ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam']
    if df.duplicated(key).any():
        raise ValueError('发现重复比赛，需核对后再生成，不自动合并。')
    df = df.sort_values(['MatchDate', 'SourceRow']).reset_index(drop=True)
    appearances = pd.concat([
        df[['Country', 'MatchDate', 'HomeTeam']].rename(columns={'HomeTeam': 'Team'}),
        df[['Country', 'MatchDate', 'AwayTeam']].rename(columns={'AwayTeam': 'Team'})])
    if appearances.duplicated(['Country', 'MatchDate', 'Team']).any():
        raise ValueError('发现同队同日多场记录，需核对后再生成。')
    return df, excluded

## 3. 填补缺失的 Elo

仅补缺失的一侧，保留原值。每次填补记录来源和评分日期。日期必须严格小于比赛日，没有历史时长限制。

完成查找后，删除任意一方 Elo 仍缺失的比赛。最终文件要求双方评分都有值。

In [3]:
def elo_lookup(raw):
    ratings = raw.copy()
    ratings['date'] = pd.to_datetime(ratings.date, format='%Y-%m-%d', errors='coerce')
    ratings['elo'] = pd.to_numeric(ratings.elo, errors='coerce')
    good = ratings[['date', 'club', 'country']].notna().all(axis=1) & np.isfinite(ratings.elo)
    ratings = ratings.loc[good].copy()
    for col in ['club', 'country']:
        ratings[col] = ratings[col].map(normalize)
    counts = ratings.groupby(['country', 'club', 'date']).elo.nunique()
    conflicts = set(counts[counts > 1].index)
    if conflicts:
        ratings = ratings.loc[[tuple(r) not in conflicts for r in ratings[['country', 'club', 'date']].itertuples(index=False, name=None)]]
    ratings = ratings.drop_duplicates(['country', 'club', 'date']).sort_values('date')
    lookup = {key: (list(group.date), group.elo.to_numpy(float))
              for key, group in ratings.groupby(['country', 'club'], sort=False)}
    return lookup, {'invalid_rating_rows': int((~good).sum()), 'conflicting_team_dates': len(conflicts)}

def find_earlier_rating(lookup, key, day):
    if key not in lookup:
        return np.nan, pd.NaT, 'team_not_found'
    dates, values = lookup[key]
    pos = bisect_left(dates, day) - 1
    if pos < 0:
        return np.nan, pd.NaT, 'no_strictly_earlier_rating'
    return float(values[pos]), dates[pos], 'filled'

def fill_elo(df, lookup):
    df = df.copy()
    for side in ['Home', 'Away']:
        col = side + 'Elo'
        original = pd.to_numeric(df[col], errors='coerce')
        original = original.where(np.isfinite(original))
        missing = original.isna()
        values = original.to_numpy(copy=True)
        sources = np.where(missing, 'missing', 'Matches.csv').astype(object)
        source_dates = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
        status = np.where(missing, '', 'original_kept').astype(object)
        for i in np.flatnonzero(missing.to_numpy()):
            key = (df.at[i, 'Country'], df.at[i, side + 'Team'])
            value, rating_date, reason = find_earlier_rating(lookup, key, df.at[i, 'MatchDate'])
            status[i] = reason
            if reason == 'filled':
                values[i] = value
                sources[i] = 'EloRatings.csv'
                source_dates.at[i] = rating_date
        df[col] = values
        df[col + 'OriginallyMissing'] = missing.to_numpy()
        df[col + 'Source'] = sources
        df[col + 'RatingDate'] = source_dates
        df[col + 'FillStatus'] = status
        # Date restrictions only concern newly filled ratings; original dates are unknown.
        filled = df[col + 'Source'].eq('EloRatings.csv')
        assert (df.loc[filled, col + 'RatingDate'] < df.loc[filled, 'MatchDate']).all()
        assert np.array_equal(df.loc[~missing, col].to_numpy(), original.loc[~missing].to_numpy())
    return df

def retain_complete_elo(df):
    """最终样本只保留双方Elo都存在的比赛；不改变已建立的历史池。"""
    valid = np.isfinite(df[['HomeElo', 'AwayElo']]).all(axis=1)
    removed = df.loc[~valid].copy()
    removed['Reason'] = 'missing_elo_after_fill'
    retained = df.loc[valid].reset_index(drop=True).copy()
    assert retained[['HomeElo', 'AwayElo']].notna().all().all()
    return retained, removed


## 4. 重算此前3场、5场积分

先生成当天全部赛前统计，再更新当天赛果；只有完整窗口才输出积分总和。

先用全部有效赛果计算历史，再过滤最终样本，避免因历史某场缺 Elo 而丢失其已知比分。

In [4]:
def rebuild_form(df):
    """最近恰好3/5场的积分总和；仅使用严格早于本场日期的比赛。"""
    df = df.copy()
    totals = {col: np.full(len(df), np.nan) for col in FORM_COLUMNS}
    observed = {col: np.zeros(len(df), dtype=np.int64) for col in FORM_COLUMNS}
    prior_counts = {side: np.zeros(len(df), dtype=np.int64) for side in ['Home', 'Away']}
    history = defaultdict(lambda: deque(maxlen=5))
    counts = defaultdict(int)
    for day, batch in df.groupby('MatchDate', sort=True):
        # 先读取当天全部赛前历史，再更新当天赛果。
        for r in batch.itertuples():
            for side in ['Home', 'Away']:
                key = (r.Country, getattr(r, side + 'Team'))
                prior = list(history[key])
                assert all(date < day for date, points in prior)
                prior_counts[side][r.Index] = counts[key]
                for k in [3, 5]:
                    col = f'Form{k}{side}'
                    n = min(k, len(prior))
                    observed[col][r.Index] = n
                    if n == k:
                        totals[col][r.Index] = sum(points for date, points in prior[-k:])
        for r in batch.itertuples():
            for side, win in [('Home', 'H'), ('Away', 'A')]:
                key = (r.Country, getattr(r, side + 'Team'))
                points = 1 if r.FTResult == 'D' else 3 if r.FTResult == win else 0
                history[key].append((day, points))
                counts[key] += 1
    for col in FORM_COLUMNS:
        df[col] = pd.array(totals[col], dtype='Int64')
        df[col + 'Count'] = observed[col]
        df[col + 'Missing'] = np.isnan(totals[col])
        k = int(col[4])
        assert df[col].isna().equals(df[col + 'Count'].lt(k))
        assert df[col].dropna().between(0, 3 * k).all()
    for side in ['Home', 'Away']:
        df[side + 'PriorMatchCount'] = prior_counts[side]
    return df

## 5. 检查边界情况

检查较早评分可以使用、同日和未来评分不能使用、已有Elo保留，以及本场赛果不影响本场Form。

In [5]:
# 小样本边界检查：原值保留、比赛当天评分排除、不限制历史时长、Form排除本场。
day = pd.Timestamp('2024-03-31')
lookup = {('ENG','A'): ([pd.Timestamp('2023-12-30'), pd.Timestamp('2023-12-31'), day], np.array([1000., 1100., 2000.])),
          ('ENG','B'): ([pd.Timestamp('2024-03-30')], np.array([1900.]))}
value, rating_date, reason = find_earlier_rating(lookup, ('ENG','A'), day)
assert value == 1100 and rating_date == pd.Timestamp('2023-12-31') and reason == 'filled'
assert find_earlier_rating({('ENG','A'): ([pd.Timestamp('2010-01-01')], np.array([1000.]))}, ('ENG','A'), day)[0] == 1000  # 很早的评分也可使用
assert find_earlier_rating({('ENG','A'): ([day], np.array([2000.]))}, ('ENG','A'), day)[2] == 'no_strictly_earlier_rating'
assert find_earlier_rating({('ENG','A'): ([day + pd.Timedelta(days=1)], np.array([2000.]))}, ('ENG','A'), day)[2] == 'no_strictly_earlier_rating'
rows = [
 ['E0','2024-01-01','A','B',1,0,'H',1200.,1500.],
 ['E0','2024-01-02','A','B',0,0,'D',1200.,1500.],
 ['E0','2024-01-03','A','B',0,1,'A',1200.,1500.],
 ['E0','2024-03-31','A','B',9,0,'H',np.nan,1500.],
 ['E0','2024-04-01','A','B',1,0,None,1200.,1500.],
]
small = pd.DataFrame(rows, columns=['Division','MatchDate','HomeTeam','AwayTeam','FTHome','FTAway','FTResult','HomeElo','AwayElo'])
base, removed = prepare_matches(small)
assert len(base) == 4 and len(removed) == 1  # 结果缺失直接删除，即使比分完整
small_filled = fill_elo(base, lookup)
assert small_filled.iloc[-1].HomeElo == 1100 and small_filled.iloc[-1].AwayElo == 1500
rebuilt = rebuild_form(small_filled)
assert rebuilt.iloc[-1].Form3Home == 4 and rebuilt.iloc[-1].Form3Away == 4
assert rebuilt.iloc[-1].Form3HomeCount == 3 and pd.isna(rebuilt.iloc[-1].Form5Home)
assert pd.isna(rebuilt.iloc[0].Form3Home) and rebuilt.iloc[0].Form3HomeCount == 0
counterfactual = small_filled.copy()
counterfactual.loc[counterfactual.index[-1], ['FTHome','FTAway','FTResult']] = [0,9,'A']
assert rebuild_form(counterfactual).iloc[-1].Form3Home == 4  # 当前比赛结果不改变其赛前Form
print('小样本边界检查通过。')

# 任意一侧Elo缺失都删除；保留双方完整的行。
probe = pd.DataFrame({'HomeElo': [1200., np.nan, 1200., np.nan],
                      'AwayElo': [1500., 1500., np.nan, np.nan]})
retained_probe, removed_probe = retain_complete_elo(probe)
assert len(retained_probe) == 1 and len(removed_probe) == 3
assert (removed_probe.Reason == 'missing_elo_after_fill').all()
print('双方Elo完整性过滤检查通过。')


小样本边界检查通过。
双方Elo完整性过滤检查通过。


/tmp/ipykernel_73963/3518001590.py:9: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  assert find_earlier_rating({('ENG','A'): ([day + pd.Timedelta(days=1)], np.array([2000.]))}, ('ENG','A'), day)[2] == 'no_strictly_earlier_rating'


## 6. 生成数据集和处理明细

In [6]:
def build_dataset():
    hashes = verify_raw()
    raw = read_raw(ROOT / 'data/raw/Matches.csv')
    elo_raw = read_raw(ROOT / 'data/raw/EloRatings.csv')
    df, excluded = prepare_matches(raw)
    lookup, rating_checks = elo_lookup(elo_raw)
    df = fill_elo(df, lookup)
    df = rebuild_form(df)
    history_pool = df.copy()
    core_excluded_count = len(excluded)
    df, elo_excluded = retain_complete_elo(history_pool)
    excluded = pd.concat([excluded, elo_excluded], ignore_index=True)
    odds = df[['OddHome', 'OddDraw', 'OddAway']].apply(pd.to_numeric, errors='coerce')
    df['OddsAvailable'] = np.isfinite(odds).all(axis=1) & odds.gt(1).all(axis=1)
    # Keep optional missing values; no train/test-dependent fitting in this raw derivative.
    out = ROOT / 'data/processed/补全缺失值.csv'
    report = ROOT / 'reports/feature_engineering'
    report.mkdir(parents=True, exist_ok=True)
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    exclusion_columns = ['SourceRow', 'Division', 'MatchDate', 'HomeTeam', 'AwayTeam',
                         'FTHome', 'FTAway', 'FTResult', 'HomeElo', 'AwayElo',
                         'HomeEloFillStatus', 'AwayEloFillStatus', 'Reason']
    excluded.reindex(columns=exclusion_columns).to_csv(
        report / 'completion_excluded_matches.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    audit = []
    for col in raw.columns:
        audit.append({'column': col, 'missing_raw_all_rows': int(raw[col].isna().sum()),
                      'missing_raw_retained_rows': int(raw.loc[df.SourceRow.to_numpy() - 2, col].isna().sum()),
                      'missing_output': int(df[col].isna().sum())})
    pd.DataFrame(audit).to_csv(report / 'completion_missing_before_after.csv', index=False)
    lineage_cols = ['SourceRow', 'Division', 'Country', 'MatchDate', 'HomeTeam', 'AwayTeam']
    for side in ['Home', 'Away']:
        lineage_cols += [side + 'Elo', side + 'EloOriginallyMissing', side + 'EloSource',
                         side + 'EloRatingDate', side + 'EloFillStatus']
    changed = history_pool.HomeEloOriginallyMissing | history_pool.AwayEloOriginallyMissing
    history_pool.loc[changed, lineage_cols].to_csv(report / 'completion_elo_lineage.csv', index=False,
                                        encoding='utf-8-sig', date_format='%Y-%m-%d')
    summary = {
        'input_rows': len(raw), 'output_rows': len(df), 'excluded_rows': len(excluded),
        'core_excluded_rows': core_excluded_count, 'elo_excluded_rows': len(elo_excluded),
        'valid_result_history_pool_rows': len(history_pool),
        'input_hashes': hashes, 'output': str(out.relative_to(ROOT)), 'output_columns': len(df.columns),
        'output_sha256': digest(out), 'date_policy': 'use MatchDate as recorded; strictly earlier dates only',
        'elo_policy': 'preserve original; fill missing side from EloRatings.csv, exact normalized country/team, strictly earlier, no age limit; drop match if either Elo remains missing',
        'form_policy': 'rebuild all four columns; total points in exactly last 3/5 prior valid matches; insufficient history NA plus count/missing flags',
        'elo': {side: {
            'originally_missing_retained': int(df[side + 'EloOriginallyMissing'].sum()),
            'filled': int(df[side + 'EloSource'].eq('EloRatings.csv').sum()),
            'still_missing': int(df[side + 'Elo'].isna().sum()),
            'status_counts': {str(k): int(v) for k, v in df[side + 'EloFillStatus'].value_counts().items()}}
            for side in ['Home', 'Away']},
        'elo_before_drop': {side: {
            'originally_missing': int(history_pool[side + 'EloOriginallyMissing'].sum()),
            'filled': int(history_pool[side + 'EloSource'].eq('EloRatings.csv').sum()),
            'still_missing': int(history_pool[side + 'Elo'].isna().sum())}
            for side in ['Home', 'Away']},
        'filled_elo_age_days': {side: {
            'min': int((df.loc[df[side + 'EloSource'].eq('EloRatings.csv'), 'MatchDate'] - df.loc[df[side + 'EloSource'].eq('EloRatings.csv'), side + 'EloRatingDate']).dt.days.min()) if df[side + 'EloSource'].eq('EloRatings.csv').any() else None,
            'max': int((df.loc[df[side + 'EloSource'].eq('EloRatings.csv'), 'MatchDate'] - df.loc[df[side + 'EloSource'].eq('EloRatings.csv'), side + 'EloRatingDate']).dt.days.max()) if df[side + 'EloSource'].eq('EloRatings.csv').any() else None
        } for side in ['Home', 'Away']},
        'both_elo_available_after': int(df[['HomeElo', 'AwayElo']].notna().all(axis=1).sum()),
        'form_missing_after': {col: int(df[col].isna().sum()) for col in FORM_COLUMNS},
        'valid_odds_rows': int(df.OddsAvailable.sum()), 'rating_checks': rating_checks,
        'raw_hashes_unchanged': verify_raw() == hashes,
    }
    (report / 'completion_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    assert len(df) + len(excluded) == len(raw)
    assert df[['HomeElo', 'AwayElo']].notna().all().all()
    return df, summary, history_pool

completed, summary, history_pool = build_dataset()

{
  "input_rows": 230557,
  "output_rows": 137603,
  "excluded_rows": 92954,
  "core_excluded_rows": 3,
  "elo_excluded_rows": 92951,
  "valid_result_history_pool_rows": 230554,
  "input_hashes": {
    "data/raw/Matches.csv": "6905ca5324df95282fefcdc2b320caed40745254055f38a3e5a768b8a073ec2b",
    "data/raw/EloRatings.csv": "29f02206a928dc04d2e46e8e56c2458d42fc591b93a767c9ce2098d9a66bca11"
  },
  "output": "data/processed/补全缺失值.csv",
  "output_columns": 69,
  "output_sha256": "b8d3edb7e5936fc2434a399f1580e35eaf69633e1f0840a71bd59c0977f58445",
  "date_policy": "use MatchDate as recorded; strictly earlier dates only",
  "elo_policy": "preserve original; fill missing side from EloRatings.csv, exact normalized country/team, strictly earlier, no age limit; drop match if either Elo remains missing",
  "form_policy": "rebuild all four columns; total points in exactly last 3/5 prior valid matches; insufficient history NA plus count/missing flags",
  "elo": {
    "Home": {
      "originally_miss

## 7. 独立复核

核对全部Elo填补候选的最近评分；抽样从历史赛果重新计算Form。检查原始文件和未使用字段保持不变。

In [7]:
# 独立从原表复核原值保留、核心缺失删除、可选列未被填补。
original = read_raw(ROOT / 'data/raw/Matches.csv')
kept = original.loc[completed.SourceRow.to_numpy() - 2].reset_index(drop=True)
core = ['Division','MatchDate','HomeTeam','AwayTeam','FTHome','FTAway','FTResult']
assert completed[core].notna().all().all()
assert original[core].isna().any(axis=1).sum() == summary['core_excluded_rows'] == 3
assert completed[['HomeElo','AwayElo']].notna().all().all()
assert len(completed) + summary['excluded_rows'] == len(original)
assert summary['elo_excluded_rows'] == int(history_pool[['HomeElo','AwayElo']].isna().any(axis=1).sum())
assert set(completed.SourceRow) == set(history_pool.loc[history_pool[['HomeElo','AwayElo']].notna().all(axis=1), 'SourceRow'])
for side in ['Home','Away']:
    col = side + 'Elo'
    original_values = pd.to_numeric(kept[col], errors='coerce')
    observed = original_values.notna()
    np.testing.assert_array_equal(completed.loc[observed,col].to_numpy(), original_values.loc[observed].to_numpy())
    is_filled = completed[col + 'Source'].eq('EloRatings.csv')
    assert original_values.loc[is_filled].isna().all()
    assert (completed.loc[is_filled,col + 'RatingDate'] < completed.loc[is_filled,'MatchDate']).all()
changed_columns = set(core + FORM_COLUMNS + ['HomeElo','AwayElo'])
for col in set(original.columns) - changed_columns:
    pd.testing.assert_series_equal(completed[col], kept[col], check_names=False)

# 抽取24场，独立查询双方此前赛果，逐项核对4个Form输入。
for row in completed.sample(24, random_state=20261028).itertuples():
    for side in ['Home','Away']:
        team = getattr(row, side + 'Team')
        previous = history_pool.loc[(history_pool.Country == row.Country) & (history_pool.MatchDate < row.MatchDate)
                                 & ((history_pool.HomeTeam == team) | (history_pool.AwayTeam == team))]
        for k in [3,5]:
            recent = previous.tail(k)
            actual = getattr(row, f'Form{k}{side}')
            assert getattr(row, f'Form{k}{side}Count') == len(recent)
            if len(recent) < k:
                assert pd.isna(actual)
            else:
                points = np.where(recent.FTResult.eq('D'), 1,
                                  np.where((recent.HomeTeam.eq(team) & recent.FTResult.eq('H'))
                                           | (recent.AwayTeam.eq(team) & recent.FTResult.eq('A')), 3, 0))
                assert actual == int(points.sum())
assert summary['raw_hashes_unchanged']
print('全量检查与抽样复核通过；原始CSV未改变。')
print('数据集：', ROOT / summary['output'])

# 独立用 merge_asof 核对每个填补候选的最近日期，排除同日或未来记录。
rating_raw = read_raw(ROOT / 'data/raw/EloRatings.csv')
rating_raw['date'] = pd.to_datetime(rating_raw['date'])
rating_raw['elo'] = pd.to_numeric(rating_raw['elo'])
for col in ['country','club']:
    rating_raw[col] = rating_raw[col].map(normalize)
for side in ['Home','Away']:
    left = history_pool.loc[history_pool[side + 'EloOriginallyMissing'],
                         ['SourceRow','MatchDate','Country',side + 'Team',side + 'Elo',side + 'EloRatingDate']].copy()
    left = left.rename(columns={side + 'Team':'Club'}).sort_values('MatchDate')
    right = rating_raw.rename(columns={'country':'Country','club':'Club','date':'ExpectedDate','elo':'ExpectedElo'}).sort_values('ExpectedDate')
    checked = pd.merge_asof(left, right, left_on='MatchDate', right_on='ExpectedDate',
                            by=['Country','Club'], direction='backward', allow_exact_matches=False)
    np.testing.assert_allclose(checked[side + 'Elo'], checked.ExpectedElo, equal_nan=True)
    assert checked[side + 'EloRatingDate'].equals(checked.ExpectedDate)
assert digest(ROOT / summary['output']) == summary['output_sha256']
checks = {'passed': True, 'output_hash_matches': True, 'raw_files_unchanged': True,
          'existing_elo_preserved': True, 'only_strictly_earlier_ratings': True,
          'latest_rating_independently_checked_with_merge_asof': True,
          'old_ratings_allowed': True, 'same_day_and_future_excluded': True,
          'optional_columns_preserved': True, 'forms_independently_checked': True,
          'input_rows': summary['input_rows'], 'output_rows': summary['output_rows'],
          'output_elo_missing_count': int(completed[['HomeElo','AwayElo']].isna().sum().sum()),
          'remaining_missing_elo_rows_removed': True, 'all_valid_results_used_for_history': True}
(ROOT / 'reports/feature_engineering/completion_checks.json').write_text(
    json.dumps(checks, ensure_ascii=False, indent=2) + '\n')
print('全部检查通过，包括每一个待填补Elo的独立匹配核对。')


全量检查与抽样复核通过；原始CSV未改变。
数据集： /home/euan_li/projects/AML/data/processed/补全缺失值.csv


全部检查通过，包括每一个待填补Elo的独立匹配核对。


## 8. 查看结果并保存数据说明

In [8]:
display(pd.DataFrame(summary['elo']).T)
display(pd.read_csv(ROOT / 'reports/feature_engineering/completion_missing_before_after.csv'))
filled = completed.HomeEloSource.eq('EloRatings.csv') | completed.AwayEloSource.eq('EloRatings.csv')
display(completed.loc[filled, ['SourceRow','MatchDate','HomeTeam','AwayTeam','HomeElo','HomeEloSource','HomeEloRatingDate','AwayElo','AwayEloSource','AwayEloRatingDate']].head(12))

readme = f"""# 补全缺失值数据集

文件：[补全缺失值.csv](补全缺失值.csv)，UTF-8 BOM 编码。

共 {summary['output_rows']:,} 场、{summary['output_columns']} 列。保留原48列，覆盖重算的四个Form与成功补上的Elo，并附来源、日期、场数和标记。

Elo只填原本缺失的一侧。按国家和规范化名称精确匹配，取严格早于比赛日最近的一条，不限制时长；同日及未来评分不参与填补。原有Elo不覆盖，原有评分的独立日期未知，因此RatingDate为空。补全后任何一侧Elo仍缺失的比赛均已从最终文件删除，双方Elo缺失数均为0。

Form3／Form5是此前最近恰好3／5场的积分总和；不足场数时为空，Count记录实际场数，Missing为True。不会把本场或同日赛果纳入自身历史。历史来自全部比分和赛果有效的比赛，包括后来因Elo缺失而未导出的历史比赛。

| 结果 | 数量 |
|---|---:|
| 原始比赛 | {summary['input_rows']:,} |
| 核心字段缺失／无效删除 | {summary['core_excluded_rows']:,} |
| Elo补全后仍缺失删除 | {summary['elo_excluded_rows']:,} |
| 合计删除 | {summary['excluded_rows']:,} |
| 保留比赛 | {summary['output_rows']:,} |
| 主队Elo成功填补 | {summary['elo']['Home']['filled']:,} |
| 客队Elo成功填补 | {summary['elo']['Away']['filled']:,} |
| 主队Elo仍缺失 | {summary['elo']['Home']['still_missing']:,} |
| 客队Elo仍缺失 | {summary['elo']['Away']['still_missing']:,} |

SourceRow是原CSV行号，首条数据为第2行；Country用于球队身份匹配。EloOriginallyMissing记录原值是否缺失，EloSource为Matches.csv、EloRatings.csv或missing，EloRatingDate记录填补使用的评分日期。FillStatus为original_kept、filled、team_not_found或no_strictly_earlier_rating。双方PriorMatchCount为此前已观察到的有效比赛总场数。

OddsAvailable表示三项普通赔率均为有限数且大于1；没有因此把赔率无效的比赛从本文件删除。

最终数据集中的双方Elo均完整。历史不足的Form和未使用技术统计等仍可能为空。删除明细在completion_excluded_matches.csv，原因missing_elo_after_fill表示补全后仍缺Elo。

复现：运行[特征工程Notebook](../../notebooks/02_features.ipynb)。检查与明细在reports/feature_engineering/。两份原始文件保持不变。

SHA-256：`{summary['output_sha256']}`。
"""
(ROOT / 'data/processed/补全缺失值.README.md').write_text(readme)
print('数据集与说明已写入：', ROOT / summary['output'])


,originally_missing_retained,filled,still_missing,status_counts
Home,4252,4252,0,"{'original_kept': 133351, 'filled': 4252}"
Away,4276,4276,0,"{'original_kept': 133327, 'filled': 4276}"


,column,missing_raw_all_rows,missing_raw_retained_rows,missing_output
0,Division,0,0,0
1,MatchDate,0,0,0
2,MatchTime,131485,98188,98188
3,HomeTeam,0,0,0
4,AwayTeam,0,0,0
5,HomeElo,88960,4252,0
6,AwayElo,89029,4276,0
7,Form3Home,1500,407,877
8,Form5Home,1500,407,1475
9,Form3Away,1500,407,876


,SourceRow,MatchDate,HomeTeam,AwayTeam,HomeElo,HomeEloSource,HomeEloRatingDate,AwayElo,AwayEloSource,AwayEloRatingDate
5458,6473,2001-09-15,QPR,Port Vale,1411.58,Matches.csv,NaT,1378.04,EloRatings.csv,2000-07-01
5459,6475,2001-09-15,Swindon,Tranmere,1368.68,EloRatings.csv,2000-07-01,1374.39,Matches.csv,NaT
5539,6599,2001-09-18,Port Vale,Swindon,1378.04,EloRatings.csv,2000-07-01,1368.68,EloRatings.csv,2000-07-01
6663,8158,2001-11-21,QPR,Swindon,1411.58,Matches.csv,NaT,1368.68,EloRatings.csv,2000-07-01
6726,8235,2001-11-24,Port Vale,Huddersfield,1378.04,EloRatings.csv,2000-07-01,1473.61,Matches.csv,NaT
7133,8744,2001-12-15,Tranmere,Port Vale,1374.39,Matches.csv,NaT,1378.04,EloRatings.csv,2000-07-01
7994,9909,2002-02-05,Swindon,Huddersfield,1368.68,EloRatings.csv,2000-07-01,1473.61,Matches.csv,NaT
8315,10347,2002-02-22,Tranmere,Swindon,1374.39,Matches.csv,NaT,1368.68,EloRatings.csv,2000-07-01
8345,10386,2002-02-23,Port Vale,QPR,1378.04,EloRatings.csv,2000-07-01,1411.58,Matches.csv,NaT
8479,10597,2002-03-02,Swindon,Port Vale,1368.68,EloRatings.csv,2000-07-01,1378.04,EloRatings.csv,2000-07-01


数据集与说明已写入： /home/euan_li/projects/AML/data/processed/补全缺失值.csv


## 9. 筛选第一版基线的共同样本，生成 cleaned 数据集

1. 双方必须具有完整最近5场历史，Form5Home、Form5Away均非空且对应Count均为5。
2. OddHome、OddDraw、OddAway必须都非空、为有限数且大于1，包含OddHome=0在内的无效记录排除。
3. 删除已确定的23列。只按以上相关条件删行，未使用的技术统计和Elo评分日期允许为空。

输出 `data/processed/cleaned.csv`，各基线模型可以在该共同样本上比较。过滤在历史计算之后执行，被排除比赛的有效赛果仍可参与之后比赛的历史。排除明细保存为 `reports/feature_engineering/cleaned_excluded_matches.csv`。


In [9]:
# 用户确定的删除名单：17列暂不使用字段 + 6列填补记录。
DROP_UNUSED_COLUMNS = [
    'MatchTime',
    'MaxHome', 'MaxDraw', 'MaxAway',
    'Over25', 'Under25', 'MaxOver25', 'MaxUnder25',
    'HandiSize', 'HandiHome', 'HandiAway',
    'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD', 'C_HTB', 'C_PHB',
]
DROP_AUDIT_COLUMNS = [
    'HomeEloOriginallyMissing', 'AwayEloOriginallyMissing',
    'HomeEloSource', 'AwayEloSource',
    'HomeEloFillStatus', 'AwayEloFillStatus',
]
DROP_COLUMNS = DROP_UNUSED_COLUMNS + DROP_AUDIT_COLUMNS
assert len(DROP_COLUMNS) == len(set(DROP_COLUMNS)) == 23
assert set(DROP_COLUMNS).issubset(completed.columns)

# 先筛选共同样本：完整最近5场 + 三项赔率有效。
form5_columns = ['Form5Home', 'Form5Away']
form5_counts = ['Form5HomeCount', 'Form5AwayCount']
odds_columns = ['OddHome', 'OddDraw', 'OddAway']
form5_ok = completed[form5_columns].notna().all(axis=1) & completed[form5_counts].eq(5).all(axis=1)
odds_numeric = completed[odds_columns].apply(pd.to_numeric, errors='coerce')
odds_ok = np.isfinite(odds_numeric).all(axis=1) & odds_numeric.gt(1).all(axis=1)
assert completed.OddsAvailable.equals(odds_ok)
keep = form5_ok & odds_ok

# 保存所有排除行，并分别标明历史不足和赔率无效，避免重复计数。
exclusion_columns = ['SourceRow', 'Division', 'MatchDate', 'HomeTeam', 'AwayTeam',
                     'Form5Home', 'Form5Away', 'Form5HomeCount', 'Form5AwayCount',
                     'OddHome', 'OddDraw', 'OddAway']
cleaned_excluded = completed.loc[~keep, exclusion_columns].copy()
cleaned_excluded['IncompleteForm5'] = ~form5_ok.loc[~keep]
cleaned_excluded['InvalidOdds'] = ~odds_ok.loc[~keep]
cleaned_excluded['Reason'] = np.select(
    [~form5_ok.loc[~keep] & ~odds_ok.loc[~keep], ~form5_ok.loc[~keep]],
    ['incomplete_form5;invalid_odds', 'incomplete_form5'], default='invalid_odds')
cleaned_excluded.to_csv(ROOT / 'reports/feature_engineering/cleaned_excluded_matches.csv',
                        index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

cleaned = completed.loc[keep].drop(columns=DROP_COLUMNS).reset_index(drop=True).copy()
assert len(cleaned) + len(cleaned_excluded) == len(completed)
assert cleaned.shape[1] == completed.shape[1] - 23 == 46
assert not set(DROP_COLUMNS).intersection(cleaned.columns)
pd.testing.assert_frame_equal(cleaned, completed.loc[keep, cleaned.columns].reset_index(drop=True))
assert cleaned[['HomeElo', 'AwayElo', 'Form5Home', 'Form5Away']].notna().all().all()
assert cleaned[form5_counts].eq(5).all().all()
cleaned_odds = cleaned[odds_columns].apply(pd.to_numeric, errors='coerce')
assert np.isfinite(cleaned_odds).all().all() and cleaned_odds.gt(1).all().all()
assert cleaned.OddsAvailable.all()

cleaned_path = ROOT / 'data/processed/cleaned.csv'
cleaned.to_csv(cleaned_path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

# 从实际写出的CSV独立计算保留条件，核对所有行及保留列的值。
intermediate_saved = pd.read_csv(ROOT / summary['output'], dtype=str, keep_default_na=False)
cleaned_saved = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False)
saved_form = intermediate_saved[form5_columns].apply(pd.to_numeric, errors='coerce')
saved_counts = intermediate_saved[form5_counts].apply(pd.to_numeric, errors='coerce')
saved_odds = intermediate_saved[odds_columns].apply(pd.to_numeric, errors='coerce')
saved_keep = (saved_form.notna().all(axis=1) & saved_counts.eq(5).all(axis=1)
              & np.isfinite(saved_odds).all(axis=1) & saved_odds.gt(1).all(axis=1))
pd.testing.assert_frame_equal(cleaned_saved, intermediate_saved.loc[saved_keep].drop(columns=DROP_COLUMNS).reset_index(drop=True))
assert set(cleaned.SourceRow).isdisjoint(set(cleaned_excluded.SourceRow))
assert set(cleaned.SourceRow) | set(cleaned_excluded.SourceRow) == set(completed.SourceRow)
assert verify_raw() == summary['input_hashes']
cleaned_summary = {
    'input': summary['output'], 'output': str(cleaned_path.relative_to(ROOT)),
    'rows_before': len(completed), 'rows_after': len(cleaned),
    'columns_before': len(completed.columns), 'columns_after': len(cleaned.columns),
    'excluded_rows': len(cleaned_excluded),
    'incomplete_form5_rows': int((~form5_ok).sum()),
    'invalid_odds_rows_before_form5_filter': int((~odds_ok).sum()),
    'both_incomplete_form5_and_invalid_odds': int((~form5_ok & ~odds_ok).sum()),
    'invalid_odds_rows_after_form5_filter': int((form5_ok & ~odds_ok).sum()),
    'nonmissing_invalid_odds_rows': int((odds_numeric.notna().all(axis=1) & ~odds_ok).sum()),
    'eligibility_rule': 'both complete last 5 matches; all three decimal odds finite and > 1',
    'removed_column_count': len(DROP_COLUMNS), 'removed_columns': DROP_COLUMNS,
    'retained_columns': cleaned.columns.tolist(),
    'home_elo_missing': int(cleaned.HomeElo.isna().sum()),
    'away_elo_missing': int(cleaned.AwayElo.isna().sum()),
    'output_sha256': digest(cleaned_path),
    'checks': {'passed': True, 'eligible_rows_and_order_preserved': True,
               'excluded_rows_accounted_for_without_overlap': True,
               'form5_complete': True, 'all_odds_finite_and_greater_than_one': True,
               'retained_values_unchanged_after_csv_roundtrip': True,
               'all_23_requested_columns_removed': True, 'raw_files_unchanged': True},
}
(ROOT / 'reports/feature_engineering/cleaned_summary.json').write_text(
    json.dumps(cleaned_summary, ensure_ascii=False, indent=2) + '\n')

cleaned_readme = f"""# cleaned 数据集

文件：[cleaned.csv](cleaned.csv)，UTF-8 BOM编码。共{len(cleaned):,}场、{cleaned.shape[1]}列。

从补全缺失值.csv中筛选双方完整最近5场、且三项普通赔率均为有限数并大于1的比赛，再删除指定23列。保留行的值和先后顺序不变。双方Elo和四个Form均完整，三项赔率完整且有效；半场、技术统计及Elo评分日期仍可能缺失。

筛选前{len(completed):,}场。先排除最近5场不足的{int((~form5_ok).sum()):,}场，再从剩余比赛中排除赔率缺失或无效的{int((form5_ok & ~odds_ok).sum()):,}场，共排除{len(cleaned_excluded):,}场。赔率无空值但数值不合法的{int((odds_numeric.notna().all(axis=1) & ~odds_ok).sum())}场也已排除。

删除明细在reports/feature_engineering/cleaned_excluded_matches.csv。这些筛选只作用于最终预测样本，历史统计仍基于原始文件中此前所有有效赛果。

## 删除列

{chr(10).join('- ' + col for col in DROP_COLUMNS)}

## 保留列

{chr(10).join('- ' + col for col in cleaned.columns)}

全部代码在[02_features.ipynb](../../notebooks/02_features.ipynb)的第9部分。填补来源等信息可按SourceRow在reports/feature_engineering/completion_elo_lineage.csv查找。半场和技术统计仍保留，当前比赛的赛后统计不能作为其赛前预测输入。

这是整理后的数据表，46列不代表全部46列都应该输入模型。

SHA-256：`{cleaned_summary['output_sha256']}`。
"""
(ROOT / 'data/processed/cleaned.README.md').write_text(cleaned_readme)

root_readme = f"""# 足球比赛分析：特征工程

运行[02_features.ipynb](notebooks/02_features.ipynb)，从上到下执行即可生成最终的[cleaned.csv](data/processed/cleaned.csv)。全部处理代码都在Notebook内；本步骤不训练预测模型。

## 当前流程

1. 删除七个核心字段任一项缺失或无效的比赛。
2. 保留已有Elo；缺失的一侧从评分表查找同一国家和球队、严格早于比赛日的最近评分，没有时长上限。
3. 补全后任意一方Elo仍缺失，就删除该场。
4. 按此前有效比赛重算四个Form字段。历史池使用全部比分和赛果有效的比赛，完整窗口不足时留空并记录实际场数。
5. 保存补全缺失值.csv中间表及处理明细。
6. 排除任意一方最近5场不足，以及任意一项普通赔率缺失、非有限数或不大于1的比赛。
7. 删除指定的17列暂不使用字段和6列填补记录，生成cleaned.csv。

## 当前结果

原始{summary['input_rows']:,}场，核心信息排除{summary['core_excluded_rows']:,}场，补全后仍缺Elo排除{summary['elo_excluded_rows']:,}场。在补全缺失值.csv基础上，再排除最近5场不足的{int((~form5_ok).sum()):,}场，以及其余赔率无效的{int((form5_ok & ~odds_ok).sum()):,}场。最终cleaned保留{len(cleaned):,}场、{cleaned.shape[1]}列；双方Elo、四个Form和三项普通赔率均完整，赔率全部大于1。未使用的技术统计和评分日期仍可为空。

[cleaned字段说明](data/processed/cleaned.README.md)列出删除与保留字段。原始两份CSV不变，来源、删除原因和检查结果在reports/feature_engineering/。

46列包含身份、目标和检查信息，不代表模型需要输入46个特征。

## 运行

使用.venv虚拟环境和AML Football (.venv)内核。从项目根目录启动：

```bash
.venv/bin/jupyter lab
```

依赖在requirements.txt，原始数据在data/raw/，输出在data/processed/。
"""
if (ROOT / 'notebooks/base_line.ipynb').exists():
    root_readme += '\n## 第一版 baseline\n\n运行[base_line.ipynb](notebooks/base_line.ipynb)：近期固定5场，历史交锋使用此前全部记录，比较M0、M1、M2、M3四个逻辑回归模型，并分析与赔率的关系。所有代码均在Notebook内。\n\n2000—2020训练，2021—2022验证，2023—2024测试；2025暂不进入本版。验证仅选择正则化强度，不选择近期窗口。\n\n结果保存在[reports/baseline/RESULTS.zh-CN.md](reports/baseline/RESULTS.zh-CN.md)，完整输入清单、验证分数、逐场测试概率、模型和检查记录位于reports/baseline/。\n'
if (ROOT / 'notebooks/time_decay.ipynb').exists():
    root_readme += '\n## 时间衰减与LSTM多任务扩展\n\n按顺序运行：\n\n1. [time_decay.ipynb](notebooks/time_decay.ipynb)：近期5场、交锋全部历史，分别按90/365天半衰期计算加权统计，训练逻辑回归。输出 `time_decay_features.csv` 和 `history_sequences.npz`。\n2. [lstm_multitask.ipynb](notebooks/lstm_multitask.ipynb)：读取相同逐场历史，联合训练LSTM和预测层；比较单任务与硬参数共享多任务。主任务预测赛果，辅助任务拟合赔率概率，赔率不进入网络输入。验证集选择辅助权重和训练轮数，三个固定seed重训。\n\n两者沿用baseline的训练/验证/测试划分与比赛样本。所有代码在Notebook内，旧数据和baseline保留。\n\n结果分别在 [时间衰减报告](reports/time_decay/RESULTS.zh-CN.md) 和 [LSTM报告](reports/lstm_multitask/RESULTS.zh-CN.md)。各目录同时保存参数、输入说明、逐场预测、配对比较和模型。\n\nLSTM使用PyTorch 2.11.0，当前环境安装CUDA 12.8版本，自动选择GPU或CPU。重新安装依赖可运行 `.venv/bin/python -m pip install -r requirements.txt`；无NVIDIA显卡的机器也可以按PyTorch官方说明改装CPU版本。\n'
if (ROOT / 'data/raw/README.md').exists():
    root_readme += '\n## 从 GitHub 克隆后运行\n\n先按 [原始数据说明](data/raw/README.md) 下载两份固定版本的 CSV，再创建环境：\n\n```bash\npython3.12 -m venv .venv\n.venv/bin/python -m pip install -r requirements.txt\n.venv/bin/python -m ipykernel install --sys-prefix --name aml-football --display-name "AML Football (.venv)"\n.venv/bin/jupyter lab\n```\n\n依次完整运行 `02_features.ipynb`、`base_line.ipynb`、`time_decay.ipynb`、`lstm_multitask.ipynb`。已执行的 Notebook 和汇总结果随代码保存；原始/处理后 CSV、历史缓存、逐场预测、模型权重和虚拟环境由 `.gitignore` 排除，在本地按此顺序重新生成。\n'
if (ROOT / 'notebooks/post_analysis.ipynb').exists():
    root_readme += '\n## 模型解释\n\n前四本Notebook完成后，运行[post_analysis.ipynb](notebooks/post_analysis.ipynb)，读取已保存模型：导出22个逻辑回归模型的完整系数，展示历史比赛的固定时间权重，并整理近期/交锋输入组的贡献及多任务损失权重。该步骤不重新训练模型。\n\n解释见[reports/post_analysis/RESULTS.zh-CN.md](reports/post_analysis/RESULTS.zh-CN.md)。系数、时间权重和损失权重含义不同，均不能直接当作特征重要性百分比。\n'
(ROOT / 'README.md').write_text(root_readme)
print(json.dumps({'数据集': str(cleaned_path), '比赛数': len(cleaned),
                  '本步骤排除比赛数': len(cleaned_excluded),
                  '删除列数': len(DROP_COLUMNS), '保留列数': len(cleaned.columns),
                  '检查通过': True}, ensure_ascii=False, indent=2))
display(cleaned.head())


{
  "数据集": "/home/euan_li/projects/AML/data/processed/cleaned.csv",
  "比赛数": 133512,
  "本步骤排除比赛数": 4091,
  "删除列数": 23,
  "保留列数": 46,
  "检查通过": true
}


,SourceRow,Division,MatchDate,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,...,Form3HomeMissing,Form5HomeCount,Form5HomeMissing,Form3AwayCount,Form3AwayMissing,Form5AwayCount,Form5AwayMissing,HomePriorMatchCount,AwayPriorMatchCount,OddsAvailable
0,411,F1,2000-09-05,Monaco,Marseille,1766.17,1678.53,7,8,4,...,False,5,False,3,False,5,False,5,5,True
1,438,F1,2000-09-06,Bastia,Lille,1671.73,1660.00,3,9,6,...,False,5,False,3,False,5,False,5,5,True
2,439,F1,2000-09-06,Guingamp,Toulouse,1560.95,1547.54,0,1,2,...,False,5,False,3,False,5,False,5,5,True
3,440,F1,2000-09-06,Lens,Auxerre,1751.70,1643.20,5,11,6,...,False,5,False,3,False,5,False,5,5,True
4,441,F1,2000-09-06,Metz,Strasbourg,1671.09,1605.71,4,5,3,...,False,5,False,3,False,5,False,5,5,True
